In [8]:
# Cell 1: Setup and Device Check
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Select GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version : {torch.__version__}")
print(f"Active Device   : {device}")

PyTorch Version : 2.11.0+cpu
Active Device   : cpu


In [9]:
# Cell 2: Load Model & Tokenizer
MODEL_NAME = "t5-small"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print("Model loaded and transferred to device successfully.")

Loading t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded and transferred to device successfully.


In [12]:
# Cell 3: Summarization Function & Formula 1 Text
def generate_summary(text: str, max_tokens: int = 50, min_tokens: int = 15) -> str:
    """
    Generates an abstractive summary of input text using T5.
    """
    input_prompt = "summarize: " + text.strip()

    inputs = tokenizer(
        input_prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_new_tokens=max_tokens,
        min_length=min_tokens,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary_text


# Formula 1 Sample Article
f1_article = """
Formula 1 motorsport relies heavily on aerodynamic efficiency, tire management, and pit stop strategy.
During a Grand Prix weekend, teams analyze gigabytes of real-time telemetry data sent from hundreds of sensors
on the car to the pit wall. Engine temperature, tire degradation, downforce levels, and fuel consumption are monitored
continuously. A single pit stop strategy error, such as mistiming a swap from wet to slick tires during changing
weather conditions, can cost a driver the victory regardless of their raw qualifying pace.
"""

print(f"Original Text Word Count: {len(f1_article.split())} words")

Original Text Word Count: 82 words


In [14]:
# Cell 4: Execute and Output Summary
summary = generate_summary(f1_article, max_tokens=40, min_tokens=10)

print("\n--- Original F1 Article ---")
print(f1_article.strip())

print("\n--- Generated Summary ---")
print(summary)
print(f"\nSummary Word Count: {len(summary.split())} words")


--- Original F1 Article ---
Formula 1 motorsport relies heavily on aerodynamic efficiency, tire management, and pit stop strategy. 
During a Grand Prix weekend, teams analyze gigabytes of real-time telemetry data sent from hundreds of sensors 
on the car to the pit wall. Engine temperature, tire degradation, downforce levels, and fuel consumption are monitored 
continuously. A single pit stop strategy error, such as mistiming a swap from wet to slick tires during changing 
weather conditions, can cost a driver the victory regardless of their raw qualifying pace.

--- Generated Summary ---
teams analyze gigabytes of real-time telemetry data sent from hundreds of sensors to the pit wall. engine temperature, tire degradation, downforce levels, and fuel consumption are monitored

Summary Word Count: 27 words
